# Strategy Benchmark — all agents, head-to-head

Cross-evaluate **every strategy** (beginner, medium and advanced tiers) against
each other at **each difficulty** (Easy / Medium / Hard).

Rules:
- Each game seats **all strategies at once** (one bot per strategy → 8 bots).
- Games are played under the three difficulty presets (different starting cash,
  market sizes, taxes, active-markets-per-round → different complexity).
- 10 games per difficulty, 10 rounds each, deterministic seeds → reproducible.
- **Score** = final balance; **profit** = balance − starting balance
  (Easy 80,000 / Medium 50,000 / Hard 30,000). Leftover inventory spoils at game over.

The last cell writes the aggregate results into `README.md`.


In [1]:
import os
import sys
import random
from collections import defaultdict

# Make the backend importable (run this notebook from the `notebook/` folder).
ROOT = os.environ.get("BUYAM_ROOT", os.path.abspath(os.path.join(os.getcwd(), "..")))
BACKEND = os.path.join(ROOT, "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

from ksell.model.difficulty import Difficulty, DifficultyConfig
from ksell.model.market_board import DICE_BASE
from ksell.model.player import Player
from ksell.model.table import Table
from ksell.pojo.user import User
from ksell.strategy import ALL_STRATEGIES, get_strategy


def market_info(idx, m):
    """Same dict shape the AI strategies receive from the API."""
    return {
        "market_index": idx,
        "name": m.location.name,
        "product": m.location.product,
        "market_fixed_price": m.market_fixed_price,
        "market_supply": m.market_supply,
        "tax_rate": m.location.tax_rate,
        "sell_entry_fee": m.sell_entry_fee,
        "capacity": m.total_qty,
        "price_history": list(m.price_history),
    }


def player_info(p):
    return {
        "username": p.username,
        "balance": p.balance,
        "inventory": [
            {
                "product": {"name": it.product.name, "price": it.product.price},
                "quantity": it.quantity,
                "avg_cost": it.avg_cost,
            }
            for it in p.inventory
        ],
    }


def action_state(round_number, total_rounds, can_buy, can_sell, max_affordable, seller_qty):
    """Minimal state dict the quantity choosers read."""
    return {
        "round_number": round_number,
        "total_rounds": total_rounds,
        "can_buy": can_buy,
        "can_sell": can_sell,
        "max_affordable": max_affordable,
        "seller_qty": seller_qty,
    }


def run_game(difficulty, roster, rounds, seed=None):
    """Play one full game with every bot in `roster` (username -> strategy key).

    Mirrors the API orchestration (routes.py): strategy phase -> turn order ->
    action phase with a fresh dice roll per trade -> end of round.
    """
    if seed is not None:
        random.seed(seed)
    diff = DifficultyConfig.from_difficulty(Difficulty(difficulty))
    table = Table(total_rounds=rounds, difficulty=diff)
    for uname in roster:
        player = Player(user=User(username=uname))
        player.balance = diff.starting_balance
        table.add_player(player)

    table.generate_markets()
    table.initialize_player_inventory()
    strategies = {u: get_strategy(k) for u, k in roster.items()}

    while table.current_round < rounds:
        num_markets = diff.sample_num_markets_per_round(len(table.markets))
        markets = table.start_round(num_markets)
        round_number = table.current_round

        # Strategy phase: every bot picks buy/sell/skip per active market.
        plans = {}
        for uname, strat in strategies.items():
            mkt = [market_info(i, m) for i, m in enumerate(markets, 1)]
            pls = [player_info(p) for p in table.players]
            choices = strat.choose_strategy(
                mkt, pls, uname, round_number=round_number, total_rounds=rounds
            )
            valid = []
            for mi, action in choices:
                if action == "sell" and table.get_player(uname).get_inventory_quantity(
                    markets[mi - 1].location.product
                ) <= 0:
                    action = "skip"
                valid.append((mi, action))
            plans[uname] = valid

        # Turn order by a per-player dice roll (highest first).
        turn_order = table.determine_turn_order()

        # Action phase: fresh dice per trade, auto quantity from the strategy.
        for player, _ in turn_order:
            uname = player.username
            strat = strategies[uname]
            for mi, action in plans[uname]:
                if action == "skip":
                    continue
                market = markets[mi - 1]
                dice_total = table.roll_dice_for_player()
                dice_price = dice_total * DICE_BASE
                if action == "buy":
                    res = table.process_market_action_buy(player, market, dice_total)
                    if not res.get("can_buy"):
                        continue
                    qty = strat.choose_buy_quantity(
                        action_state(round_number, rounds, True, False, res["max_affordable"], 0)
                    )
                    table.execute_buy_at_market_price(
                        player, market, min(qty, res["max_affordable"])
                    )
                else:  # sell
                    fee = table.pay_sell_entry_fee(player, market)
                    if not fee["success"]:
                        continue
                    res = table.process_market_action_sell(player, market, dice_total)
                    if not res.get("can_sell"):
                        continue
                    qty = strat.choose_sell_quantity(
                        action_state(round_number, rounds, False, True, 0, res["seller_qty"])
                    )
                    table.execute_market_auto_buy(
                        player, market, min(qty, res["seller_qty"]), dice_price
                    )

        table.end_round(markets)

    # Game over: leftover inventory spoils, score = final cash balance.
    return {
        p.username: {
            "balance": round(p.balance, 2),
            "profit": round(p.balance - diff.starting_balance, 2),
            "spoiled_units": sum(it.quantity for it in p.inventory),
        }
        for p in table.players
    }

In [2]:
DIFFICULTIES = ("easy", "medium", "hard")
GAMES = 10  # games per difficulty
ROUNDS = 10  # rounds per game

# Every strategy sits at the table in every game.
ROSTER = {name: name for name in ALL_STRATEGIES}

# raw[(difficulty, strategy)] = list of per-game results
raw = defaultdict(list)
for gi in range(GAMES):
    for di, diff in enumerate(DIFFICULTIES):
        results = run_game(diff, ROSTER, ROUNDS, seed=di * 100_000 + gi)
        order = sorted(results, key=lambda u: results[u]["balance"], reverse=True)
        rank = {u: i + 1 for i, u in enumerate(order)}
        for u, r in results.items():
            raw[(diff, u)].append(
                {
                    "profit": r["profit"],
                    "balance": r["balance"],
                    "spoiled": r["spoiled_units"],
                    "rank": rank[u],
                }
            )


def aggregate(diff):
    rows = []
    for u in ALL_STRATEGIES:
        items = raw[(diff, u)]
        n = len(items)
        rows.append(
            (
                u,
                sum(x["profit"] for x in items) / n,
                sum(x["balance"] for x in items) / n,
                sum(1 for x in items if x["rank"] == 1),
                sum(x["rank"] for x in items) / n,
                sum(x["spoiled"] for x in items) / n,
            )
        )
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows


def money(x):
    return ("+" if x >= 0 else "") + f"{int(round(x)):,}"


def pct(x):
    return f"{int(round(x * 100))}%"


TITLES = {
    "easy": "Easy — beginner markets (2–4 active per round)",
    "medium": "Medium — standard markets (1–3 active per round)",
    "hard": "Hard — tight markets (1–2 active per round)",
}

lines = []
lines.append("## Strategy Benchmark")
lines.append("")
lines.append(
    f"> Auto-generated by `notebook/strategy_benchmark.ipynb`. All **{len(ALL_STRATEGIES)} strategies** share every table "
    f"(one bot each), averaged over **{GAMES} games per difficulty × {ROUNDS} rounds** (deterministic seeds)."
)
lines.append(
    "> **Profit** = final balance − starting balance (Easy 80,000 / Medium 50,000 / Hard 30,000). "
    "**Win rate** = share of games finished 1st. **Avg spoiled** = units left to spoil at game over."
)

for diff in DIFFICULTIES:
    lines.append("")
    lines.append(f"### {TITLES[diff]}")
    lines.append("")
    lines.append("| Strategy | Avg profit | Avg final | Win rate | Avg rank | Avg spoiled |")
    lines.append("|---|---|---|---|---|---|")
    for u, profit, balance, wins, rank, spoiled in aggregate(diff):
        lines.append(
            f"| {u} | {money(profit)} | {balance:,.0f} | {pct(wins / GAMES)} | {rank:.1f} | {spoiled:.1f} |"
        )

lines.append("")
lines.append("### Overall — wins across all difficulties")
lines.append("")
lines.append("| Strategy | Wins | Win rate |")
lines.append("|---|---|---|")
total_games = GAMES * len(DIFFICULTIES)
overall = sorted(
    (
        (u, sum(1 for d in DIFFICULTIES for x in raw[(d, u)] if x["rank"] == 1))
        for u in ALL_STRATEGIES
    ),
    key=lambda r: r[1],
    reverse=True,
)
for u, wins in overall:
    lines.append(f"| {u} | {wins} | {pct(wins / total_games)} |")

MARKDOWN = chr(10).join(lines)

# Render the results as rich Markdown inside the notebook.
from IPython.display import Markdown, display

display(Markdown(MARKDOWN))

## Strategy Benchmark

> Auto-generated by `notebook/strategy_benchmark.ipynb`. All **8 strategies** share every table (one bot each), averaged over **10 games per difficulty × 10 rounds** (deterministic seeds).
> **Profit** = final balance − starting balance (Easy 80,000 / Medium 50,000 / Hard 30,000). **Win rate** = share of games finished 1st. **Avg spoiled** = units left to spoil at game over.

### Easy — beginner markets (2–4 active per round)

| Strategy | Avg profit | Avg final | Win rate | Avg rank | Avg spoiled |
|---|---|---|---|---|---|
| endgame | +5,946 | 85,946 | 40% | 2.2 | 14.8 |
| expectedvalue | +5,629 | 85,629 | 20% | 2.1 | 14.5 |
| conservativetrader | +3,072 | 83,072 | 10% | 2.9 | 23.3 |
| arbitrageur | +379 | 80,379 | 0% | 4.6 | 29.5 |
| marketsniper | -5,079 | 74,921 | 20% | 4.4 | 32.0 |
| buylowsellhigh | -10,097 | 69,903 | 10% | 5.0 | 38.2 |
| random | -52,418 | 27,582 | 0% | 6.9 | 133.6 |
| aggressivebuyer | -77,844 | 2,156 | 0% | 7.9 | 221.9 |

### Medium — standard markets (1–3 active per round)

| Strategy | Avg profit | Avg final | Win rate | Avg rank | Avg spoiled |
|---|---|---|---|---|---|
| endgame | +8,123 | 58,123 | 60% | 1.7 | 3.2 |
| expectedvalue | +6,083 | 56,083 | 20% | 2.3 | 6.8 |
| conservativetrader | +2,716 | 52,716 | 0% | 3.7 | 14.8 |
| arbitrageur | +0 | 50,000 | 0% | 4.8 | 20.0 |
| buylowsellhigh | -4,857 | 45,143 | 10% | 4.7 | 20.8 |
| marketsniper | -8,128 | 41,872 | 0% | 5.0 | 18.2 |
| random | -15,305 | 34,695 | 10% | 6.0 | 42.7 |
| aggressivebuyer | -35,545 | 14,455 | 0% | 7.8 | 71.1 |

### Hard — tight markets (1–2 active per round)

| Strategy | Avg profit | Avg final | Win rate | Avg rank | Avg spoiled |
|---|---|---|---|---|---|
| expectedvalue | +4,824 | 34,824 | 40% | 1.9 | 1.6 |
| endgame | +4,469 | 34,469 | 40% | 1.8 | 1.8 |
| conservativetrader | +1,666 | 31,666 | 0% | 3.3 | 6.3 |
| arbitrageur | +0 | 30,000 | 0% | 4.4 | 10.0 |
| marketsniper | -8,482 | 21,518 | 0% | 5.9 | 13.3 |
| buylowsellhigh | -9,182 | 20,818 | 10% | 5.4 | 14.4 |
| random | -12,470 | 17,530 | 10% | 6.0 | 21.7 |
| aggressivebuyer | -17,092 | 12,908 | 0% | 7.3 | 23.9 |

### Overall — wins across all difficulties

| Strategy | Wins | Win rate |
|---|---|---|
| endgame | 14 | 47% |
| expectedvalue | 8 | 27% |
| buylowsellhigh | 3 | 10% |
| random | 2 | 7% |
| marketsniper | 2 | 7% |
| conservativetrader | 1 | 3% |
| aggressivebuyer | 0 | 0% |
| arbitrageur | 0 | 0% |

In [3]:
import pathlib
import re

from IPython.display import Markdown, display

readme = pathlib.Path(ROOT) / "README.md"
start_marker = "<!-- benchmark:start -->"
end_marker = "<!-- benchmark:end -->"
nl = chr(10)
block = start_marker + nl + nl + MARKDOWN + nl + nl + end_marker

text = readme.read_text()
if start_marker in text and end_marker in text:
    text = re.sub(re.escape(start_marker) + r".*?" + re.escape(end_marker), block, text, flags=re.S)
else:
    anchor = "## Project Structure"
    if anchor in text:
        text = text.replace(anchor, block + nl + nl + anchor, 1)
    else:
        text = text.rstrip() + nl + nl + block + nl
readme.write_text(text)
display(Markdown(f"**README updated:** `{readme}`"))

**README updated:** `/home/eak/Documents/AI/Game/ksell/buyam-sellam/README.md`